<a href="https://colab.research.google.com/github/rorisDS/workshop_ai_agents/blob/develop/notebooks_es/TuPrimerAgenteIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tu Primer Agente de IA

En este notebook vamos a construir y ejecutar un **Agente de IA extremadamente simple**, con un único objetivo: **entender cómo funciona un agente por dentro**.

La idea no es crear un sistema inteligente complejo ni “útil” en términos prácticos, sino **aislar los elementos mínimos** que hacen que un LLM deje de ser un simple modelo de conversación y pase a comportarse como un **agente**:

* Un **modelo de lenguaje** que razona sobre una petición.
* Un conjunto de **herramientas (tools)** que puede utilizar para actuar.
* Un **prompt de sistema** que define su rol y sus restricciones.
* Un **bucle de decisión** en el que el modelo decide cuándo y cómo usar esas herramientas.

Para ello, implementaremos un agente capaz de **resolver operaciones matemáticas básicas** utilizando herramientas explícitas (suma, multiplicación, cuadrado), en lugar de calcular directamente el resultado.

Este ejemplo nos permitirá observar:

- Cómo el agente interpreta una petición del usuario.
- Cómo decide **qué herramientas invocar** para resolverla.
- Cómo **encadena múltiples llamadas a tools** para llegar a una respuesta final.
- Qué información intercambia el agente con cada herramienta durante la ejecución.

A partir de este punto, el mismo patrón se puede extender a agentes mucho más complejos: acceso a bases de datos, APIs externas, sistemas RAG, planificación multi-paso o incluso interacción entre múltiples agentes.

## Instalación de librerías

In [1]:
!pip install langchain==1.2.7
!pip install langchain-core==1.2.7
!pip install langchain-openai==1.1.7  # Para usar modelos de OpenAI
!pip install langchain-google-genai==4.2.0  # Para usar modelos de Google (Gemini)
!pip install ddgs==9.10.0
!pip install langchain-community==0.4.1

# Limpia output
from IPython.display import clear_output
clear_output()

In [2]:
# Codigo auxiliar desarrollado para facilitar la visualizacion de los mensajes en el agente
import os

project_path = "/content/workshop_ai_agents"

if os.path.exists(project_path) == False:
  !git clone https://github.com/rorisDS/workshop_ai_agents

import sys
sys.path.append(project_path)

from utils.agent_message_pretty_debug import PrettyDebug

Cloning into 'workshop_ai_agents'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 7), reused 12 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 651.48 KiB | 5.71 MiB/s, done.
Resolving deltas: 100% (7/7), done.


## LLM

In [3]:
# Use Google Colab Secrets
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    pass

* Conectando a un modelo de OpenAI

   - Crear API Key: https://platform.openai.com/docs/quickstart
   - Seleccionar un modelo: https://platform.openai.com/docs/models

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o",  # Seleccionamos un modelo por su keyword
    temperature=0,
    max_tokens=None,
    # other params...
    callbacks=[PrettyDebug()]   # <--- Verbose
    )

* Conectando a un modelo de Google
   - Crear API Key: https://ai.google.dev/gemini-api/docs/api-key
   - Seleccionar un modelo: https://ai.google.dev/gemini-api/docs/models

In [4]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.0,
    max_tokens=None,
    # other params...
    callbacks=[PrettyDebug()]   # <--- Verbose
)

## Tools

In [5]:
from langchain_core.tools import tool

# setup the tools
@tool
def suma(a: int, b: int) -> int:
    """Suma dos numeros."""
    return a + b


@tool
def multiplicacion(a: int, b: int) -> int:
    """Multiplica 2 numeros."""
    return a * b


@tool
def cuadrado(a: int) -> int:
    """Calcula el cuadrado de un numero."""
    return a * a

## Agente

Un agente de IA es la combinación coordinada de un modelo de lenguaje, un conjunto de mensajes y una serie de herramientas, todo ello orquestado por una lógica de decisión.

<table style="background:white;">
<tr>
<td>
<img src="https://raw.githubusercontent.com/rorisDS/workshop_ai_agents/refs/heads/develop/images/agent_graph.png" width="400"/>
</td>
</tr>
</table>

Una vez que entiendes cada uno de estos componentes (LLM, mensajes, tools…), la lógica interna de un agente resulta sorprendentemente simple y siempre sigue el mismo patrón:

- El modelo interpreta la intención del usuario.
- Evalúa qué herramientas pueden ayudar a resolver la tarea.
- Ejecuta una o varias llamadas a esas herramientas.
- Integra los resultados para generar una respuesta final.

Es decir, un agente no es magia: es un bucle de razonamiento y acción.

LangChain proporciona la abstracción [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents), que encapsula este ciclo por nosotros, permitiéndonos centrarnos en definir qué sabe hacer el agente (sus tools) y cómo debe comportarse (su prompt), sin tener que implementar manualmente toda la lógica de orquestación.

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[suma, multiplicacion, cuadrado],
    system_prompt="""Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia.
Devuelve solo el resultado de la operacion."""
)

In [7]:
from langchain.messages import HumanMessage

result = agent.invoke(
    {
        "messages": [
            HumanMessage("Cual es el resultado de: ((3+3)-2^2)*5")
        ]
    }
)


▶ LLM START

Message:
System: Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia.
Devuelve solo el resultado de la operacion.
Human: Cual es el resultado de: ((3+3)-2^2)*5

■ LLM END
 ==> ▶ suma(b=3, a=3)


▶ LLM START

Message:
System: Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia.
Devuelve solo el resultado de la operacion.
Human: Cual es el resultado de: ((3+3)-2^2)*5
AI: [{'name': 'suma', 'args': {'b': 3, 'a': 3}, 'id': '8010f5ce-d27b-43fe-8583-a809423dd73e', 'type': 'tool_call'}]
 ▶ suma(b=3, a=3) → 6

■ LLM END
 ==> ▶ cuadrado(a=2)


▶ LLM START

Message:
System: Eres un asistente de matematicas.
Utiliza las tools para responder las preguntas. Si no tienes una tool para responder una pregunta, deja constancia.
Devuelve solo el resultado de la operacion.
Human: Cual es el resultado 

Ahora la respuesta del `agente` es un diccionario con la lista de mensajes que se han generado a lo largo del ciclo de ejecucion

In [8]:
result

{'messages': [HumanMessage(content='Cual es el resultado de: ((3+3)-2^2)*5', additional_kwargs={}, response_metadata={}, id='0212fb5b-ffff-4314-8396-9ae1d25bf825'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'suma', 'arguments': '{"b": 3, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'8010f5ce-d27b-43fe-8583-a809423dd73e': 'ErEKCq4KAb4+9vsmtrDmfWFPhETSN9rxHAI8LpEP82mXHMXeH9KuE/OWSyPzl0BAw8BhUDAkEbvP8uVEZJhWO58DuJ/rExsFN1b2kboxA1eJWZU5giLEZbjjeMKzrDN4/Jce5HmdjHMVtWd76vdPZHBf9C5TKBR7MkpVvaEPT7Z8/EvxdtbF+YQEz45PMk1rUVA4JEBxkxLc4YSHZI2NozMOtAsDdyzGWPWwQO7H9a0OYStskGVv6URsedM+yoM1Zv2yHkn8ErbD6Xorz5e6B1FPhfMQCJ54Un7tO9phnEUuluyKXrIWWyVJDdOIEb5ryiIa5FAI/6OFNp8WBzdaK7ng0MPiTdSf6BHbRao83YRpB9eNz/FS0OUPGVdKLbyBnFEUt2NKNeUn4Dp5TPE2e6pqr6XWVJK3IzoEe6E/trgEA2zP4cABkv36jBhlNLUMFDcQomfeeLw2Qx4LEVtv0aEaUiTqcUaabpV3OBxhhdYnFNTgzt1lV2B3m6rGM8ZiFgYLheyj6cPnupYLl2Gw2IgJYbIk+QyZ4JaN1AcVD0VGrSJp1OcZ0mk7GUeLi4oMjQJpvFx+fO2K24Hqr8d7nDhtvyWWTyd5x5pmYxMcFVpzdEKLDRh0d8ZfnDt

In [9]:
# Print the conversation
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")

================================ Human Message =================================

Cual es el resultado de: ((3+3)-2^2)*5
================================== Ai Message ==================================

[]
Tool Calls:
  suma (8010f5ce-d27b-43fe-8583-a809423dd73e)
 Call ID: 8010f5ce-d27b-43fe-8583-a809423dd73e
  Args:
    b: 3
    a: 3
================================= Tool Message =================================
Name: suma

6
================================== Ai Message ==================================

[]
Tool Calls:
  cuadrado (e5b9a702-7b33-4c7e-ad5f-fa9780126f21)
 Call ID: e5b9a702-7b33-4c7e-ad5f-fa9780126f21
  Args:
    a: 2
================================= Tool Message =================================
Name: cuadrado

4
================================== Ai Message ==================================

[]
Tool Calls:
  suma (a41d2cee-6a2f-42b3-a9ff-f0a2fac6e2cd)
 Call ID: a41d2cee-6a2f-42b3-a9ff-f0a2fac6e2cd
  Args:
    a: 6
    b: -4
================================= Tool Me

In [ ]:
for message in result["messages"]:
    print(type(message), " : ", message.content or message.tool_calls)

<class 'langchain_core.messages.human.HumanMessage'>  :  Cual es el resultado ((3+3)-2^2)*5?
<class 'langchain_core.messages.ai.AIMessage'>  :  [{'name': 'suma', 'args': {'b': 3, 'a': 3}, 'id': '5430199d-7153-481c-82cf-27ae3e4b204e', 'type': 'tool_call'}, {'name': 'cuadrado', 'args': {'a': 2}, 'id': '61d9f97f-fc39-4c1c-8636-f1626c7f7161', 'type': 'tool_call'}]
<class 'langchain_core.messages.tool.ToolMessage'>  :  6
<class 'langchain_core.messages.tool.ToolMessage'>  :  4
<class 'langchain_core.messages.ai.AIMessage'>  :  [{'name': 'suma', 'args': {'a': 6, 'b': -4}, 'id': '37948c1d-431c-408c-8a86-e7a0010af2a5', 'type': 'tool_call'}]
<class 'langchain_core.messages.tool.ToolMessage'>  :  2
<class 'langchain_core.messages.ai.AIMessage'>  :  [{'name': 'multiplicacion', 'args': {'a': 2, 'b': 5}, 'id': 'a9b3070e-6b73-4319-bf26-0e202f552007', 'type': 'tool_call'}]
<class 'langchain_core.messages.tool.ToolMessage'>  :  10
<class 'langchain_core.messages.ai.AIMessage'>  :  [{'type': 'text', 't

Nuestra respuesta final estara en el último de los mensajes de la cola

In [ ]:
print(f"Respuesta: {result["messages"][-1].content[0]['text']}")

Respuesta: 10


## Conclusiones

En este notebook hemos construido un agente deliberadamente simple, pero suficiente para entender qué significa realmente crear un **Agente de IA**.

A diferencia de una interacción clásica con un LLM, aquí el modelo no se limita a generar texto: recibe una petición, analiza el problema, decide qué acciones son necesarias y utiliza herramientas externas para llegar a una respuesta.

Aunque el ejemplo es trivial (operaciones matemáticas), el flujo que hemos visto es exactamente el mismo que se utiliza en agentes reales:

- El modelo interpreta la intención del usuario.
- Evalúa qué herramientas pueden ayudar.
- Ejecuta una o varias llamadas a dichas herramientas.
- Integra los resultados en una respuesta final.

Este patrón —*razonar → actuar → observar → responder*— es el núcleo de cualquier sistema basado en agentes.

<table style="background:white;">
<tr>
<td>
<img src="https://raw.githubusercontent.com/rorisDS/workshop_ai_agents/refs/heads/develop/images/agent_graph.png" width="400"/>
</td>
</tr>
</table>

También hemos visto que gran parte del comportamiento del agente no viene del código, sino del **prompt de sistema** y de la descripción de las herramientas. Es ahí donde se define qué puede hacer el agente, cuándo debe usar una tool y cómo debe responder.

Este ejemplo demuestra que un agente no es una entidad “mágica”: es un LLM orquestado alrededor de decisiones, contexto y capacidades externas. A partir de esta base mínima, es posible construir sistemas mucho más sofisticados, pero siempre apoyados en los mismos principios fundamentales que acabamos de explorar.
